# ITAI 4376 — Week 5 throwaway experiment
**Team Carrillo (individual) · Elizabeth Carrillo · Option B mock**

This notebook is course work for a self-checkout freeze log. It is not a store pilot. No live POS, no customer names, no loyalty IDs, no cameras.

**Kill assumption:**kiosk fields I can generate without a store system can flag a freeze *without* using the label as an input.

If that is false, the charter is a story. This notebook is the cheap test.

## 0. Setup

In [1]:
import sys
print(sys.version)

try:
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import confusion_matrix
    from sklearn.model_selection import train_test_split
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "numpy"])
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import confusion_matrix
    from sklearn.model_selection import train_test_split

import json, time
from pathlib import Path

SEED = 4376
rng = np.random.default_rng(SEED)
print("seed", SEED)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
seed 4376


## 1. Staged events

Four freeze types from the floor list in the Week of 8–13 September log:
- will not scan
- bag-scale mismatch
- age-restricted item still open
- unexpected item in bag

Clears are not all clean. Some have heavy-produce scale noise or a slow bagger.
Twelve percent of freezes are messy on purpose.

`is_freeze` and `freeze_kind` are labels. They do **not** go into the rules or the model.

In [2]:
def make_row(is_freeze: bool, freeze_kind):
    scan_ok = True
    scale_delta_g = float(rng.normal(8, 12))
    age_check_needed = False
    unexpected_bag = False
    idle_s = float(max(0.2, rng.normal(1.4, 0.6)))
    items_in_tx = int(rng.integers(1, 18))
    attendant_busy = int(rng.random() < 0.35)

    if is_freeze:
        if freeze_kind == "wont_scan":
            scan_ok = False
            idle_s = float(max(3.0, rng.normal(8.0, 2.5)))
        elif freeze_kind == "bag_scale":
            scale_delta_g = float(rng.choice([-1, 1]) * rng.uniform(80, 420))
            idle_s = float(max(2.5, rng.normal(6.0, 2.0)))
        elif freeze_kind == "age_restricted":
            age_check_needed = True
            idle_s = float(max(4.0, rng.normal(12.0, 3.0)))
        elif freeze_kind == "unexpected_bag":
            unexpected_bag = True
            scale_delta_g = float(rng.uniform(60, 350))
            idle_s = float(max(3.0, rng.normal(7.0, 2.0)))
        if rng.random() < 0.12:
            scan_ok = True
            if freeze_kind != "bag_scale":
                scale_delta_g = float(rng.normal(15, 20))
    else:
        if rng.random() < 0.15:
            scale_delta_g = float(rng.uniform(40, 90))
        if rng.random() < 0.08:
            idle_s = float(rng.uniform(4.0, 9.0))
        if rng.random() < 0.05:
            age_check_needed = True

    return {
        "scan_ok": int(scan_ok),
        "scale_delta_g": round(scale_delta_g, 1),
        "age_check_needed": int(age_check_needed),
        "unexpected_bag": int(unexpected_bag),
        "idle_s": round(idle_s, 2),
        "items_in_tx": items_in_tx,
        "attendant_busy": attendant_busy,
        "freeze_kind": freeze_kind if is_freeze else "none",
        "is_freeze": int(is_freeze),
    }

kinds = ["wont_scan", "bag_scale", "age_restricted", "unexpected_bag"]
rows = []
for i in range(60):
    rows.append(make_row(True, kinds[i % 4]))
for _ in range(60):
    rows.append(make_row(False, None))

df = pd.DataFrame(rows)
df.insert(0, "event_id", [f"E{i:03d}" for i in range(1, len(df) + 1)])
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df.to_csv("staged_events_sample.csv", index=False)
print(df.shape)
print(df["is_freeze"].value_counts().to_dict())
df.head(8)

(120, 10)
{0: 60, 1: 60}


,event_id,scan_ok,scale_delta_g,age_check_needed,unexpected_bag,idle_s,items_in_tx,attendant_busy,freeze_kind,is_freeze
0,E085,1,11.7,0,0,1.12,7,0,none,0
1,E041,0,1.3,0,0,8.50,13,0,wont_scan,1
2,E056,1,288.9,0,1,5.75,13,0,unexpected_bag,1
3,E066,1,11.4,0,0,8.86,17,0,none,0
4,E091,1,84.3,0,0,1.21,12,0,none,0
5,E053,0,-1.0,0,0,4.49,14,0,wont_scan,1
6,E036,1,253.8,0,1,7.42,10,1,unexpected_bag,1
7,E027,1,21.0,1,0,15.86,1,1,age_restricted,1


## 2. Split

80 train / 40 holdout, stratified. Holdout should land near 20 freeze / 20 clear.
This is **not** the charter's official 40 + 40 test.

In [3]:
FEATURES = [
    "scan_ok", "scale_delta_g", "age_check_needed", "unexpected_bag",
    "idle_s", "items_in_tx", "attendant_busy",
]

train, test = train_test_split(df, test_size=40, random_state=16, stratify=df["is_freeze"])
print("train", len(train), "holdout", len(test))
print("holdout freeze/clear", int((test.is_freeze==1).sum()), int((test.is_freeze==0).sum()))

train 80 holdout 40
holdout freeze/clear 20 20


## 3. Approach A — hand rules



In [4]:
def rule_flag(row) -> int:
    if row["scan_ok"] == 0:
        return 1
    if abs(row["scale_delta_g"]) >= 75:
        return 1
    if row["age_check_needed"] == 1 and row["idle_s"] >= 5:
        return 1
    if row["unexpected_bag"] == 1:
        return 1
    if row["idle_s"] >= 10:
        return 1
    return 0

def eval_preds(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    freeze_n = int((y_true == 1).sum())
    clear_n = int((y_true == 0).sum())
    return {
        "freeze_n": freeze_n,
        "clear_n": clear_n,
        "hits": int(tp),
        "misses": int(fn),
        "false_flags": int(fp),
        "true_clears": int(tn),
    }

pred_a = test.apply(rule_flag, axis=1).to_numpy()
res_a = eval_preds(test["is_freeze"].to_numpy(), pred_a)
print("Approach A (rules) holdout:", res_a)

Approach A (rules) holdout: {'freeze_n': 20, 'clear_n': 20, 'hits': 19, 'misses': 1, 'false_flags': 0, 'true_clears': 20}


## 4. Approach B — logistic regression

Same seven fields. Label is the target, never a feature.
A shallow tree is recorded only as a check. It is not the selected path.

In [5]:
X_train = train[FEATURES].to_numpy()
y_train = train["is_freeze"].to_numpy()
X_test = test[FEATURES].to_numpy()
y_test = test["is_freeze"].to_numpy()

clf = LogisticRegression(max_iter=400, random_state=SEED)
t0 = time.perf_counter()
clf.fit(X_train, y_train)
print("fit seconds", round(time.perf_counter() - t0, 4))

pred_b = clf.predict(X_test)
res_b = eval_preds(y_test, pred_b)
print("Approach B (logreg) holdout:", res_b)
print("coefficients")
for name, w in zip(FEATURES, clf.coef_[0]):
    print(f"  {name:18s} {w:+.3f}")

tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
tree.fit(X_train, y_train)
res_t = eval_preds(y_test, tree.predict(X_test))
print("B-alt depth-3 tree holdout:", res_t)

fit seconds 0.5813
Approach B (logreg) holdout: {'freeze_n': 20, 'clear_n': 20, 'hits': 17, 'misses': 3, 'false_flags': 2, 'true_clears': 18}
coefficients
  scan_ok            -0.582
  scale_delta_g      +0.006
  age_check_needed   +0.405
  unexpected_bag     +1.007
  idle_s             +0.851
  items_in_tx        -0.062
  attendant_busy     +0.557
B-alt depth-3 tree holdout: {'freeze_n': 20, 'clear_n': 20, 'hits': 20, 'misses': 0, 'false_flags': 2, 'true_clears': 18}


## 5. Time-to-notice in the mock

Charter criterion 3: from staged freeze inject to the attendant log, median ≤ 3 seconds.
This measures the machine loop, not walking speed.

In [6]:
latencies_ms = []
log_rows = []
for i in range(20):
    ev = make_row(True, kinds[i % 4])
    t_inj = time.perf_counter()
    flag = rule_flag(ev)
    dt = (time.perf_counter() - t_inj) * 1000
    latencies_ms.append(dt)
    log_rows.append({"seq": i, "flag": flag, "idle_s": ev["idle_s"], "ms": round(dt, 4)})

print("median ms", round(float(np.median(latencies_ms)), 4))
print("p95 ms", round(float(np.percentile(latencies_ms, 95)), 4))
print("under 3 seconds?", bool(np.median(latencies_ms) < 3000))
pd.DataFrame(log_rows).head()

median ms 0.0011
p95 ms 0.0022
under 3 seconds? True


,seq,flag,idle_s,ms
0,0,1,6.26,0.0021
1,1,1,6.62,0.0037
2,2,1,8.44,0.0020
3,3,1,6.48,0.0011
4,4,1,5.66,0.0007


## 6. Save evidence

In [7]:
out = test.copy()
out["pred_rule"] = pred_a
out["pred_logreg"] = pred_b
out.to_csv("holdout_predictions.csv", index=False)

summary = {
    "n_total": int(len(df)),
    "n_holdout": int(len(test)),
    "approach_A_rules_holdout": res_a,
    "approach_B_logreg_holdout": res_b,
    "approach_B_tree_holdout": res_t,
    "median_rule_latency_ms": float(np.median(latencies_ms)),
    "note": "Holdout is ~20/20, not the official 40+40 charter set. Generator and rules share an author.",
}
Path("experiment_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("wrote staged_events_sample.csv, holdout_predictions.csv, experiment_summary.json")

{
  "n_total": 120,
  "n_holdout": 40,
  "approach_A_rules_holdout": {
    "freeze_n": 20,
    "clear_n": 20,
    "hits": 19,
    "misses": 1,
    "false_flags": 0,
    "true_clears": 20
  },
  "approach_B_logreg_holdout": {
    "freeze_n": 20,
    "clear_n": 20,
    "hits": 17,
    "misses": 3,
    "false_flags": 2,
    "true_clears": 18
  },
  "approach_B_tree_holdout": {
    "freeze_n": 20,
    "clear_n": 20,
    "hits": 20,
    "misses": 0,
    "false_flags": 2,
    "true_clears": 18
  },
  "median_rule_latency_ms": 0.0011109999888958555,
  "note": "Holdout is ~20/20, not the official 40+40 charter set. Generator and rules share an author."
}
wrote staged_events_sample.csv, holdout_predictions.csv, experiment_summary.json
